# ArchX - Fine-Tuning LoRA sur AMD Developer Cloud

Ce notebook fine-tune le modèle **Gemma** (9B ou 4B selon la VRAM disponible) 
avec **LoRA** (Low-Rank Adaptation) sur notre dataset d'analyses architecturales.

**Pipeline :**
1. Détection du GPU AMD et de la VRAM
2. Installation des dépendances
3. Chargement du dataset
4. Chargement du modèle (bf16 ou 4-bit selon VRAM)
5. Configuration LoRA
6. Entraînement avec SFTTrainer
7. Sauvegarde de l'adaptateur
8. Test d'inférence


## Cellule 1 — Installation des dépendances

In [ ]:
# Installation des bibliothèques nécessaires pour le fine-tuning sur AMD ROCm
# Note : On n'installe PAS bitsandbytes car il est instable sur ROCm.
#        On utilisera bf16 natif (parfaitement supporté sur MI200X/MI300X).

import subprocess
import sys

packages = [
    'transformers>=4.47',
    'trl>=0.12',
    'peft>=0.13',
    'accelerate>=1.1',
    'datasets',
    'huggingface_hub',
]

for pkg in packages:
    print(f'Installation de {pkg}...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

print('\n✅ Toutes les dépendances sont installées.')

## Cellule 2 — Détection du GPU AMD

In [ ]:
import subprocess
import torch

print('=' * 60)
print('        DÉTECTION DU GPU AMD')
print('=' * 60)

# --- Infos via rocm-smi (outil AMD natif) ---
print('\n📊 Informations rocm-smi :')
try:
    result = subprocess.run(['rocm-smi', '--showproductname', '--showmeminfo', 'vram'],
                           capture_output=True, text=True, timeout=10)
    print(result.stdout)
except FileNotFoundError:
    print('  ⚠️  rocm-smi non trouvé — peut-être amd-smi à la place.')
    try:
        result = subprocess.run(['amd-smi', 'metric', '--vram'],
                               capture_output=True, text=True, timeout=10)
        print(result.stdout)
    except Exception:
        print('  ⚠️  amd-smi aussi non trouvé. On continue avec torch.')

# --- Infos via PyTorch ---
print('\n🔥 PyTorch + ROCm :')
if torch.cuda.is_available():
    n_gpus = torch.cuda.device_count()
    print(f'  GPU(s) disponibles : {n_gpus}')
    for i in range(n_gpus):
        props = torch.cuda.get_device_properties(i)
        vram_gb = props.total_memory / (1024**3)
        print(f'  GPU {i} : {props.name}')
        print(f'    VRAM totale : {vram_gb:.1f} GB')
        print(f'    Multiprocesseurs : {props.multi_processor_count}')

    # VRAM libre sur le GPU 0
    free_mem, total_mem = torch.cuda.mem_get_info(0)
    free_gb  = free_mem  / (1024**3)
    total_gb = total_mem / (1024**3)
    print(f'\n  VRAM GPU 0 — Libre : {free_gb:.1f} GB / Total : {total_gb:.1f} GB')

    # Décision automatique du mode de chargement du modèle
    if total_gb >= 40:
        LOAD_MODE = 'bf16'
        print(f'\n  ✅ VRAM suffisante ({total_gb:.0f} GB) → Chargement en bf16 (recommandé sur AMD).')
    else:
        LOAD_MODE = '4bit'
        print(f'\n  ⚠️  VRAM limitée ({total_gb:.0f} GB) → Chargement en 4-bit (bitsandbytes requis).')
else:
    print('  ❌ Aucun GPU détecté par PyTorch ! Vérifiez votre installation ROCm.')
    LOAD_MODE = 'cpu'

print(f'\n  MODE SÉLECTIONNÉ : {LOAD_MODE}')
print('=' * 60)

## Cellule 3 — Connexion Hugging Face

In [ ]:
# Gemma est un modèle "gated" (accès restreint).
# Vous devez :
#   1. Avoir un compte Hugging Face
#   2. Avoir accepté les conditions d'utilisation sur : https://huggingface.co/google/gemma-2-9b-it
#   3. Créer un token d'accès sur : https://huggingface.co/settings/tokens

from huggingface_hub import login
import os

# Méthode 1 : Token dans une variable d'environnement (recommandé)
hf_token = os.environ.get('HF_TOKEN', '')

# Méthode 2 : Saisie manuelle si la variable n'est pas définie
if not hf_token:
    hf_token = input('🔑 Entrez votre token Hugging Face (hf_...) : ').strip()

login(token=hf_token, add_to_git_credential=False)
print('✅ Connexion Hugging Face réussie.')

## Cellule 4 — Chargement du dataset

In [ ]:
import json
from pathlib import Path
from datasets import Dataset

# ─────────────────────────────────────────────────────────
# CONFIGURATION : Chemin vers votre fichier training_data.jsonl
# ─────────────────────────────────────────────────────────
# Option A : Fichier uploadé directement dans JupyterLab
DATASET_PATH = './training_data.jsonl'

# Option B : Depuis Google Drive (décommentez si besoin)
# from google.colab import drive
# drive.mount('/content/drive')
# DATASET_PATH = '/content/drive/MyDrive/architect-insight/training_data.jsonl'
# ─────────────────────────────────────────────────────────

def load_jsonl(path: str) -> list[dict]:
    """Charge un fichier JSONL ligne par ligne."""
    records = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

def format_as_chat(record: dict) -> str:
    """
    Convertit un exemple du dataset en format de conversation chat.
    C'est le même format que celui vu à l'entraînement — la cohérence est cruciale.
    """
    inp = record['input']
    out = record['output']

    # Construction du message utilisateur à partir des métriques
    metrics = inp.get('metrics', {})
    anti_patterns = inp.get('anti_patterns', [])

    user_msg = (
        f"Analyse architecturale requise pour un projet {inp.get('sector', 'inconnu')}.\n"
        f"Équipe : {inp.get('team_size', '?')} personnes.\n"
        f"Métriques clés :\n"
        f"  - Couplage : {metrics.get('coupling_score', '?')}\n"
        f"  - Cohésion : {metrics.get('cohesion_score', '?')}\n"
        f"  - Complexité cyclomatique (moy.) : {metrics.get('avg_cyclomatic_complexity', '?')}\n"
        f"  - Couverture de tests (%) : {metrics.get('test_coverage_estimate', '?')}\n"
        f"  - Ratio bugfix hotspot : {metrics.get('top_hotspot_bugfix_ratio', '?')}\n"
        f"Anti-patterns détectés : {', '.join(anti_patterns) if anti_patterns else 'Aucun'}\n\n"
        f"Fournis une analyse complète et une recommandation en JSON structuré."
    )

    # Réponse attendue du modèle (la sortie du Teacher)
    assistant_msg = json.dumps(out, ensure_ascii=False)

    # Format chat standard (Gemma Instruct)
    return (
        f"<start_of_turn>user\n{user_msg}<end_of_turn>\n"
        f"<start_of_turn>model\n{assistant_msg}<end_of_turn>"
    )

# Chargement
print(f'📂 Chargement du dataset depuis : {DATASET_PATH}')
raw_data = load_jsonl(DATASET_PATH)
print(f'  → {len(raw_data)} exemples chargés.')

# Formatage
formatted = [{'text': format_as_chat(r)} for r in raw_data]
dataset = Dataset.from_list(formatted)

# Séparation train / validation (90% / 10%)
split = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split['train']
eval_dataset  = split['test']

print(f'  → Entraînement : {len(train_dataset)} exemples')
print(f'  → Validation   : {len(eval_dataset)} exemples')
print('\n📄 Exemple de texte formaté (extrait) :')
print(train_dataset[0]['text'][:500] + '...')

## Cellule 5 — Chargement du modèle

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# ─────────────────────────────────────────────────────────
# CONFIGURATION : Choisissez votre modèle de base
# ─────────────────────────────────────────────────────────
# Option A : Gemma 2 9B (recommandé — meilleur équilibre qualité/vitesse sur MI200X+)
BASE_MODEL = 'google/gemma-2-9b-it'

# Option B : Gemma 4B (si VRAM très limitée)
# BASE_MODEL = 'google/gemma-2-2b-it'

# Option C : Gemma 4 12B (si vous avez MI300X avec 192 Go)
# BASE_MODEL = 'google/gemma-3-12b-it'  # Vérifier le nom exact sur HF
# ─────────────────────────────────────────────────────────

print(f'🤖 Chargement du modèle : {BASE_MODEL}')
print(f'   Mode : {LOAD_MODE}')

# Chargement du tokenizer
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'  # Nécessaire pour SFTTrainer
print('✅ Tokenizer chargé.')

# Chargement du modèle selon la VRAM disponible
if LOAD_MODE == 'bf16':
    # Mode recommandé sur AMD — bf16 est natif sur les cartes MI series
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        torch_dtype=torch.bfloat16,
        device_map='auto',           # Répartit automatiquement sur le(s) GPU(s)
        trust_remote_code=True,
        attn_implementation='eager', # 'eager' est plus stable que 'flash_attention_2' sur ROCm
    )
    print('✅ Modèle chargé en bf16.')

elif LOAD_MODE == '4bit':
    # Mode 4-bit pour GPU avec moins de VRAM (nécessite bitsandbytes)
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.bfloat16,
    )
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        quantization_config=bnb_config,
        device_map='auto',
        trust_remote_code=True,
    )
    print('✅ Modèle chargé en 4-bit (QLoRA).')

else:
    raise EnvironmentError('Aucun GPU disponible. Fine-tuning impossible sans GPU.')

# Affichage de la VRAM utilisée après chargement
torch.cuda.synchronize()
used_gb = torch.cuda.memory_allocated(0) / (1024**3)
print(f'   VRAM utilisée après chargement du modèle : {used_gb:.1f} GB')

## Cellule 6 — Configuration LoRA

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

# Préparation du modèle pour le fine-tuning en 4-bit (si applicable)
if LOAD_MODE == '4bit':
    model = prepare_model_for_kbit_training(model)

# Activation du gradient checkpointing pour réduire la consommation mémoire
# (Permet d'entraîner des modèles plus larges en échange d'un peu de vitesse)
model.gradient_checkpointing_enable()

# ─────────────────────────────────────────────────────────
# CONFIGURATION LORA
# r=16, alpha=32 : bon équilibre entre puissance d'adaptation et généralisation
# Augmenter r pour un dataset plus grand (ex: r=32 pour 1000+ exemples)
# ─────────────────────────────────────────────────────────
lora_config = LoraConfig(
    r=16,                        # Rang de la décomposition — plus élevé = plus de paramètres appris
    lora_alpha=32,               # Facteur de scaling (généralement 2x le rang)
    target_modules=[             # Modules de l'attention sur lesquels appliquer LoRA
        'q_proj',                # Query projection
        'k_proj',                # Key projection
        'v_proj',                # Value projection
        'o_proj',                # Output projection
        'gate_proj',             # MLP gate
        'up_proj',               # MLP up
        'down_proj',             # MLP down
    ],
    lora_dropout=0.05,           # Régularisation pour éviter le surapprentissage
    bias='none',                 # Pas de biais entraînable (standard)
    task_type=TaskType.CAUSAL_LM # Modèle de langage causal (génération de texte)
)

# Application de la configuration LoRA au modèle
model = get_peft_model(model, lora_config)

# Affichage du nombre de paramètres entraînables vs total
model.print_trainable_parameters()
# Résultat attendu : seulement ~1-2% des paramètres sont entraînables avec LoRA
print('\n✅ LoRA configuré et appliqué.')

## Cellule 7 — Entraînement avec SFTTrainer

In [ ]:
from trl import SFTTrainer, SFTConfig
from transformers import TrainingArguments

# Dossier de sauvegarde du modèle fine-tuné
OUTPUT_DIR = './architect-insight-lora'

# ─────────────────────────────────────────────────────────
# HYPERPARAMÈTRES D'ENTRAÎNEMENT
# Ajustés pour AMD ROCm et pour notre taille de dataset (~160-250 exemples)
# ─────────────────────────────────────────────────────────
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,

    # ── Epochs et taille de batch ──
    num_train_epochs=3,              # 3 epochs = bon compromis pour ~200 exemples
    per_device_train_batch_size=2,   # Augmentez si vous avez beaucoup de VRAM (ex: 4 ou 8)
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,   # Batch effectif = 2 * 4 = 8 exemples

    # ── Optimiseur ──
    optim='adamw_torch',             # 'adamw_torch' est stable sur ROCm
    learning_rate=2e-4,              # Taux d'apprentissage standard pour LoRA
    lr_scheduler_type='cosine',      # Scheduler cosinus (décroissance douce)
    warmup_ratio=0.03,               # 3% de warmup pour stabiliser le début
    weight_decay=0.01,               # Légère régularisation L2

    # ── Précision et mémoire ──
    bf16=True,                       # bf16 est le mode natif des GPU AMD MI series
    fp16=False,                      # Ne pas activer les deux en même temps
    gradient_checkpointing=True,     # Économie mémoire
    dataloader_num_workers=0,        # 0 = utilise le thread principal (plus stable sur ROCm)

    # ── Longueur de séquence ──
    max_seq_length=2048,             # Longueur max des tokens (nos exemples font ~600-1200 tokens)
    dataset_text_field='text',       # Nom du champ texte dans notre dataset
    packing=False,                   # False = plus simple, True = plus rapide mais plus complexe

    # ── Évaluation et sauvegarde ──
    eval_strategy='steps',
    eval_steps=50,                   # Évaluer toutes les 50 étapes
    save_strategy='steps',
    save_steps=50,
    save_total_limit=2,              # Garder seulement les 2 meilleurs checkpoints
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',

    # ── Logs ──
    logging_steps=10,
    logging_dir=f'{OUTPUT_DIR}/logs',
    report_to='none',                # 'none' = pas de Weights & Biases ni TensorBoard externe

    # ── Reproductibilité ──
    seed=42,
)

# Création du trainer
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
)

print('🏋️ Démarrage de l\'entraînement...')
print(f'   Exemples d\'entraînement : {len(train_dataset)}')
print(f'   Epochs : {training_args.num_train_epochs}')
print(f'   Batch effectif : {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}')
print('─' * 50)

# Lancement de l'entraînement
train_result = trainer.train()

print('\n' + '─' * 50)
print('✅ Entraînement terminé !')
print(f'   Loss finale d\'entraînement : {train_result.training_loss:.4f}')
print('   (Un loss < 0.8 est un bon signe. Un loss > 2.0 indique un problème.)')

## Cellule 8 — Sauvegarde de l'adaptateur LoRA

In [ ]:
import os

# Sauvegarde de l'adaptateur LoRA (SEULEMENT les poids supplémentaires, ~100 Mo)
# Ce n'est PAS le modèle complet — c'est uniquement le "delta" appris par LoRA.
# Pour utiliser le modèle, il faudra charger Gemma de base + cet adaptateur.

ADAPTER_SAVE_PATH = './architect-insight-lora-final'

print(f'💾 Sauvegarde de l\'adaptateur LoRA vers : {ADAPTER_SAVE_PATH}')
trainer.model.save_pretrained(ADAPTER_SAVE_PATH)
tokenizer.save_pretrained(ADAPTER_SAVE_PATH)

# Vérification
saved_files = os.listdir(ADAPTER_SAVE_PATH)
print(f'   Fichiers sauvegardés : {saved_files}')

# Taille de l'adaptateur
total_size = sum(
    os.path.getsize(os.path.join(ADAPTER_SAVE_PATH, f))
    for f in saved_files
) / (1024**2)
print(f'   Taille totale de l\'adaptateur : {total_size:.1f} MB')

print('\n✅ Adaptateur LoRA sauvegardé avec succès !')
print('   Pour réutiliser le modèle fine-tuné plus tard :')
print(f'   model = PeftModel.from_pretrained(base_model, \'{ADAPTER_SAVE_PATH}\')')

## Cellule 9 — Test d'inférence

In [ ]:
import json
import torch
from transformers import pipeline

print('🧪 Test d\'inférence avec le modèle fine-tuné...')
print('─' * 60)

# Mettre le modèle en mode évaluation
model.eval()

# ─────────────────────────────────────────────────────────
# SCÉNARIO DE TEST : Un projet Django avec des problèmes
# (métriques similaires à celles que le modèle a vues à l'entraînement)
# ─────────────────────────────────────────────────────────
test_metrics = {
    'sector': 'e-commerce',
    'team_size': 8,
    'metrics': {
        'coupling_score': 7.2,           # Couplage élevé — mauvais signe
        'cohesion_score': 3.4,           # Faible cohésion — spaghetti code
        'avg_cyclomatic_complexity': 12, # Complexité très élevée
        'test_coverage_estimate': 18.0,  # Tests insuffisants
        'top_hotspot_bugfix_ratio': 0.45 # 45% des commits corrigent des bugs
    },
    'anti_patterns': ['God Class', 'Long Method', 'Circular Dependency']
}

# Construction du prompt (même format que l'entraînement)
inp = test_metrics
metrics = inp['metrics']
anti_patterns = inp.get('anti_patterns', [])

user_msg = (
    f"Analyse architecturale requise pour un projet {inp.get('sector', 'inconnu')}.\n"
    f"Équipe : {inp.get('team_size', '?')} personnes.\n"
    f"Métriques clés :\n"
    f"  - Couplage : {metrics.get('coupling_score', '?')}\n"
    f"  - Cohésion : {metrics.get('cohesion_score', '?')}\n"
    f"  - Complexité cyclomatique (moy.) : {metrics.get('avg_cyclomatic_complexity', '?')}\n"
    f"  - Couverture de tests (%) : {metrics.get('test_coverage_estimate', '?')}\n"
    f"  - Ratio bugfix hotspot : {metrics.get('top_hotspot_bugfix_ratio', '?')}\n"
    f"Anti-patterns détectés : {', '.join(anti_patterns) if anti_patterns else 'Aucun'}\n\n"
    f"Fournis une analyse complète et une recommandation en JSON structuré."
)

prompt = f'<start_of_turn>user\n{user_msg}<end_of_turn>\n<start_of_turn>model\n'

print('📝 Métriques envoyées au modèle :')
print(json.dumps(test_metrics, indent=2, ensure_ascii=False))
print('\n🤖 Réponse du modèle fine-tuné :')
print('─' * 60)

# Tokenisation et génération
inputs = tokenizer(prompt, return_tensors='pt').to('cuda')

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=1024,    # Longueur max de la réponse générée
        temperature=0.1,        # Très bas = réponse déterministe et précise (bon pour JSON)
        do_sample=True,
        repetition_penalty=1.1, # Évite les répétitions
        pad_token_id=tokenizer.eos_token_id,
    )

# Décodage de la réponse (en ignorant le prompt d'entrée)
response = tokenizer.decode(
    outputs[0][inputs['input_ids'].shape[1]:],
    skip_special_tokens=True
)

print(response)

# Vérification que la réponse est un JSON valide
print('\n─' * 60)
try:
    parsed = json.loads(response.strip())
    print('✅ La réponse est un JSON valide !')
    print(f'   Recommandation : {parsed.get("recommendation", "?")}')  
except json.JSONDecodeError as e:
    print(f'⚠️  La réponse n\'est pas un JSON parfait : {e}')
    print('   (Normal lors des premières inférences — le modèle peut parfois ajouter du texte avant le JSON)')

## Cellule 10 — (Optionnel) Benchmark avant/après

Cette cellule compare les réponses du modèle **avant** et **après** le fine-tuning sur un même jeu de test.
C'est une démonstration puissante pour le jury du concours AMD !


In [ ]:
# Pour charger le modèle de base SANS fine-tuning et comparer les réponses :

# from transformers import AutoModelForCausalLM, AutoTokenizer
# from peft import PeftModel
# import torch

# base_model = AutoModelForCausalLM.from_pretrained(
#     BASE_MODEL, torch_dtype=torch.bfloat16, device_map='auto'
# )
# finetuned_model = PeftModel.from_pretrained(base_model, './architect-insight-lora-final')

# Générez avec les deux modèles sur le même prompt et comparez.
# Un bon fine-tuning devrait produire :
#   - Base model    → Texte libre, souvent non-JSON, verbeux
#   - Fine-tuné     → JSON structuré, précis, respectant vos garde-fous

print('Décommentez le code ci-dessus pour faire le benchmark avant/après.')
print('C\'est une slide de démonstration très percutante pour le jury AMD !')